*The companion to [what kind of game is this](https://www.kaggle.com/code/destbreso/kaggriculture-what-kind-of-game-is-this). That one asks what the game is. This one asks what problem we are entitled to solve, given everything the series has already established, and lets that answer pick the method, instead of picking a method and defending it afterwards.*

---

## 0. The ground we are standing on

Nothing below is new here. Each line is a result from an earlier notebook in this series, and together they are what makes the rest of this one possible.

**The field is a population, not a set of independent optimisers.** [The leaderboard is a habitat gradient](https://www.kaggle.com/code/destbreso/the-leaderboard-is-a-habitat-gradient), what you meet at one rating band is not what you meet at another. [Everyone is playing the same opening](https://www.kaggle.com/code/destbreso/everyone-is-playing-the-same-opening), unrelated teams emit identical actions, turn for turn, deep into the episode. That is common descent, not convergence.

**A route has conserved and variable regions, and they can be located.** [Mutants at the top](https://www.kaggle.com/code/destbreso/mutants-at-the-top-genetic-potential-of-a-replay), three of the top ten hold one construction plan across every episode while varying their trading on up to 140 turns. A conservation map of an agent is a thing you can compute.

**The environment turns over, and fast.** [The leaderboard has a fossil record](https://www.kaggle.com/code/destbreso/the-leaderboard-has-a-fossil-record). A lineage takes a third of all seats in two days, holds three, and is gone in five. **Drift is not measurement noise here. It is the environment.**

**And there is no dominant strategy to find.** [What kind of game is this](https://www.kaggle.com/code/destbreso/kaggriculture-what-kind-of-game-is-this). The game is symmetric and zero-sum in its outcome, so its value is zero and **no winning strategy exists**: a strategy beating everyone from seat 0 would have to beat itself from seat 1. An *optimal* agent exists by von Neumann's theorem; a *winning* one does not. The same notebook measures the opponent as invisible until turn 136, about a fifth of the season committed before the question "who am I playing" can even be asked.

---

## 0b. What follows from that, in four steps

This is the chain. Each step is a consequence of the line above it, and the last one is the problem this notebook then formalises.

**Step 1, "find the best agent" is not a well-posed goal.** No dominant strategy exists. What is well posed is *be adapted to the ecosystem you are actually in*.

**Step 2. An adaptation has a shelf life.** The ecosystem turns over in about a week. So the quantity worth maximising is not the quality of one agent but **the rate at which we can produce adapted ones**, and the cost of each attempt.

**Step 3, adaptation has a substrate.** The field is a population of related routes with conserved and variable regions we can locate. Mutating a proven high-fitness route on its identified variable channel is cheaper than designing from nothing by orders of magnitude, that is the whole content of the mutants notebook, turned from a description into a method.

**Step 4, so the problem is: given a route with a located variable channel, search that channel efficiently, and measure the result against a field that is moving while you measure it.**

**On reinforcement learning, stated carefully because the careless version is wrong.** Nothing here says RL could not find a strong policy. What it cannot do is *guarantee* the policy is strong for the state of the game actually being played, because there is no dominant strategy to converge to and the opponent distribution moves faster than a training run. A policy trained against a snapshot is adapted to that snapshot.

**And this notebook does not claim its approach is best.** It claims the approach is *reasonable and derivable* from the results above. In a game with no winning strategy, "best approach" is not a claim anyone can make honestly; efficiency in the search for adaptations is a claim you can defend and then measure.

## 0c. Why the problem has to be constrained before it can be attacked

Take the game unconstrained. On each turn an agent chooses an action for the farmer, an action for each hired hand, and a list of market orders.

In [ ]:
# The engine's own unit operations, from kaggriculture.py. Quantities and
# legality are ignored, so this is a floor on the branching factor rather than
# an estimate of it.
MOVES = ["N", "S", "E", "W"]
TILE_OPS = ["PASS", "WATER", "HARVEST", "FERTILIZE", "DIG", "BUILD_COOP",
            "BUILD_PASTURE", "FEED", "COLLECT_FERTILIZER", "CARE", "DROP"]
CROPS = ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON"]
CARRIABLE = ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON", "EGG", "MILK",
             "WOOL", "FERTILIZER", "GOOSE", "COW", "SHEEP"]

per_unit = (len(MOVES) + len(TILE_OPS) + len(CROPS)          # PLANT x crop
            + len(CARRIABLE) + len(CARRIABLE))               # PICKUP, PLACE
UNITS = 13          # the farmer plus up to twelve hands
TURNS = 720

print(f"distinct operations for ONE unit, ignoring quantities : {per_unit}")
print(f"units acting on the same turn                          : {UNITS}")
print(f"joint unit-action space per turn                       : {per_unit}^{UNITS}"
      f" = {per_unit**UNITS:.2e}")
print(f"turns in a season                                      : {TURNS}")
print(f"\nand the market list is a further choice of up to 10 orders on top of that.")

That number is not the problem by itself, large spaces are searched all the time. **The problem is its shape**, and three properties of it are measured rather than assumed.

**It has no usable gradient and delayed credit.** A tile planted on day 4 pays on day 14, and a purchase that silently fails costs a quarter of the board for the rest of the season.

**Its parameters are coupled, so no separable search converges.** We built the unconstrained thing first: a policy engine of 1,303 lines and 49 parameters. It rates **650.9** and **wins 0 of 24** against a plain recorded replay. Worse, its own tuning instrument was invalid: on the version measured, **36 of 70 parameters change more than half of all turns**, so moving one face of a coupled block is a false negative by construction. One-at-a-time perturbation cannot work on that object.

**And that is not an artefact of the big version.** It reappears on the eight-parameter seller in section 5 of this notebook: the best value **any single axis reaches on its own is 8 wins of 20, while one point that moves three axes together reaches 12**. Coordinate methods plateau here because the plateau is a projection artefact, not a feature of the surface.

**So the constraint is not a convenience. It is what makes the problem well posed.** Freezing the construction plan removes the coupled combinatorial half, and the mutants notebook is precisely what licenses freezing *that* half rather than some other, because it measured which half the field itself holds fixed.

**What it buys, exactly:** what remains is separable across nine goods, concave in each, with a transition function we hold in closed form. Sections 1 and 2 show that; sections 3 to 6 are what is left to decide once it is true.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 120,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
})
TEAL, AMBER, INK, GREY, RED = "#0F766E", "#B45309", "#0F172A", "#94A3B8", "#B91C1C"

## 1. Freeze the plan, and write down what is actually left

An action is three fields: `farmer`, `hands`, `market`. Take the first two from a recording and hold them fixed. What remains is, on each turn $t$ and for each good $g$, a quantity $q_{g,t} \ge 0$ to sell.

**The engine gives the price in closed form**, which is the fact everything else rests on. For a good $g$ whose market inventory is $v$,

$$
p_g(v) \;=\; \max\!\Big(1,\; \operatorname{round}\big(b_g - a_g\, f_g(v - I_0)\big)\Big),
\qquad
a_g \;=\; \frac{\tau_g\, b_g}{f_g(T_g)}
$$

with $b_g$ the base price, $I_0 = 10{,}000$ the starting inventory, $f_g \in \{\mathrm{id},\ \sqrt{\cdot},\ \log(1+\cdot),\ (\cdot)^2\}$ the shape, and $\tau_g, T_g$ the engine's own calibration constants. Inventory starts at $I_0$ for every good and **rises with every unit either farm sells**.

Two consequences follow immediately, and they are the two that matter.

**Revenue is concave in quantity.** Selling $q$ units into an inventory $v$ earns

$$
R_g(v, q) \;=\; \sum_{k=0}^{q-1} p_g(v+k)
$$

and $p_g$ is non-increasing, so $R_g(v,\cdot)$ is concave: each additional unit is worth no more than the one before it.

**The goods are separable.** Each has its own inventory $v_g$, so the nine sub-problems interact only through three shared constraints. Writing $\theta$ for the seller's parameters and $m$ for the plan's own demands, the whole thing is

$$
\max_{\{q_{g,t}\}} \;\; \sum_{t=1}^{720} \sum_{g \in G} R_g\!\big(v_{g,t},\, q_{g,t}\big)
$$

$$
\text{s.t.}\quad
\underbrace{\textstyle\sum_g h_{g,t} \le 100}_{\text{shed}},
\qquad
\underbrace{c_t \ge \kappa_t}_{\text{the plan's bill this turn}},
\qquad
\underbrace{h_{g,t} - q_{g,t} \ge \rho_{g,t}}_{\text{inputs the plan will pick up}},
\qquad
\underbrace{|\mathcal{O}_t| \le 10}_{\text{orders per turn}}
$$

where $h_{g,t}$ is what the shed holds, $c_t$ the bank, $\kappa_t$ what the recorded plan is about to spend, and $\rho_{g,t}$ what its future `PICKUP`s will claim.

**That third constraint is not decoration, and we found it the hard way.** The engine settles market orders in list order, so a sale placed before a purchase in the same list funds it inside the turn. Drop $\kappa_t$ from the problem and the plan you froze quietly stops being the plan you are executing: a purchase that cannot afford itself is a silent no-op, with no error and no trace in the action stream.

In [ ]:
# Engine constants, kaggriculture.py. The glut branch, which is where a season
# spends almost all of itself: inventory only rises when someone sells.
I0, FLOOR = 10_000, 1
CURVE = {"WHEAT": ("log", 0.20, 400, 25), "CARROT": ("sqrt", 0.70, 450, 35),
         "TOMATO": ("sqrt", 0.60, 200, 60), "STRAWBERRY": ("linear", 1.60, 100, 120),
         "MELON": ("sq", 3.60, 300, 250), "EGG": ("log", 0.20, 332, 50),
         "MILK": ("linear", 1.60, 122, 160), "WOOL": ("sq", 3.20, 105, 200),
         "FERTILIZER": ("linear", 0.40, 200, 100)}

def shape(f, x):
    x = np.maximum(x, 0.0)
    return {"linear": x, "sq": x * x, "sqrt": np.sqrt(x),
            "log": np.log1p(x)}[f]

def price(item, sold):
    f, target, T, base = CURVE[item]
    amp = target * base / shape(f, T)
    return np.maximum(FLOOR, np.round(base - amp * shape(f, sold)))

sold = np.arange(0, 401)
rows = []
for item in CURVE:
    p = price(item, sold)
    floored = np.argmax(p <= FLOOR) if (p <= FLOOR).any() else len(sold)
    rows.append({"good": item, "base price": CURVE[item][3],
                 "units to reach the $1 floor": int(floored) if floored else None,
                 "total revenue to the floor": int(p[:floored].sum()) if floored else None})
purse = pd.DataFrame(rows).sort_values("units to reach the $1 floor")
display(purse)

In [ ]:
fig, ax = plt.subplots(figsize=(8.6, 4.6))
order = purse.good.tolist()
cmap = plt.cm.viridis(np.linspace(0.05, 0.9, len(order)))
for c, item in zip(cmap, order):
    p = price(item, sold)
    ax.plot(sold, p, lw=2, color=c, label=item)
    k = int(np.argmax(p <= FLOOR)) if (p <= FLOOR).any() else None
    if k:
        ax.plot([k], [p[k]], "o", ms=5, color=c)
        ax.annotate(f"{item.title()} {k}", (k, 6), rotation=90, fontsize=7.5,
                    color=c, ha="center", va="bottom")
ax.set_xlim(0, 400); ax.set_ylim(0, 265)
ax.set_xlabel("units of this good sold into the market, by BOTH farms together")
ax.set_ylabel("price of the next unit ($)")
ax.set_title("Every good has a cliff, and it is closer than it looks", loc="left")
ax.legend(ncol=3, frameon=False, fontsize=8, loc="upper right")
plt.tight_layout(); plt.show()

**Wool is worthless after 59 units. Milk after 76. Melon after 158.** Two farms each planting a melon quadrant produce about 300 units into a purse of 158, so most of that crop is grown for a dollar a unit.

*Reconciling with the earlier notebook, which puts melon's floor at 188:* that figure counts what can be **sold**, and this one counts **net supply**. The town centre eats one melon a day, 30 over the season, and 158 + 30 = 188. Same fact, one with the town's appetite added back.

That single figure does most of the work in this notebook. It says the money in each good is a **finite purse** that both farms draw from, and that the interesting decision is not *how much to sell*, you will sell it all eventually, since inventory scores nothing at the end, but **who reaches the top of each curve first**.

### But the curve is static and a season is not, so we checked which goods it actually binds for

The chart above is computed on a market nobody is draining. Real seasons have town shops consuming every four turns all year, which holds the price up, and [Rayk Kretzschmar's ladder notebook](https://www.kaggle.com/code/raykkretzschmar/kaggriculture-rank-your-agent) argues from that this column is the wrong one to optimise: what decides your realised price is how many shops want your product, not how steep its glut curve is.

That is a claim about this figure, so here is the measurement. Final market inventory relative to the starting $I_0$, over **120 real ladder episodes** of the ten highest-rated submissions:

| good | median final inventory | glutted in |
|---|---|---|
| FERTILIZER | **+454** | **100 %** of episodes |
| MELON | **+152** | **100 %** |
| MILK | **+76** | 78 % |
| WOOL | **+59** | 74 % |
| STRAWBERRY | +24 | 59 % |
| TOMATO | −192 | **0 %** |
| EGG | −210 | 2 % |
| CARROT | −303 | 3 % |
| WHEAT | −492 | 2 % |

**He is right about four goods of nine, and the split is total rather than gradual.** Wheat, carrot, egg and tomato finish *below* where they started in 97 to 100 % of episodes: two farms cannot out-supply the town, their price never leaves the region at or above base, and their distance-to-the-floor never applies. For those four the right quantity is base price times shop demand, as he says.

**And for the other five the curve binds absolutely.** Fertilizer and melon glut in every single episode. The detail that settles the mechanism is milk and wool: their median final inventory is **+76 and +59**, which are *exactly* the floors the curve computes above. The field sells those two precisely to the floor and then stops.

So the finite-purse framing is right for melon, fertilizer, wool, milk and strawberry, and it is the wrong picture for the four the town drains faster than anyone can supply.

**And it points at the algorithm, though the theorem is narrower than the problem.** A separable objective, concave in each coordinate, under **one** shared resource constraint is the classical resource-allocation problem, and there the greedy rule that spends the next unit where the marginal value is highest is *optimal* rather than merely sensible (Ibaraki & Katoh, 1988).

**Our problem has four constraints, not one**, and the cash-flow one couples across turns, so greedy is not provably optimal here. What the theorem buys is a principled starting shape, marginal-value ordering with a stopping rule, instead of a hand-set rate; whether it is good enough is an empirical question, and sections 5 and 6 are where it gets answered rather than assumed.

## 2. Do we have the model? Because that is the question that decides everything else

Reinforcement learning is the right tool when you have a simulator, a large state space, and **no model**, when the only way to know what an action does is to try it. Every one of those conditions has to hold. Here the third does not.

In [ ]:
known = pd.DataFrame([
    ("transition function", "EXACT",
     "our replay harness reproduces 40 of 40 recorded ladder banks to the dollar"),
    ("price function", "EXACT, closed form",
     "reproduced to within $2 at every inventory checked, with a fallback when it disagrees"),
    ("our own future production", "DETERMINISTIC given the plan and the seed", ""),
    ("weed spawn, shop draw", "STOCHASTIC but SEEDED", "and the shops are observable"),
    ("the opponent's future supply", "UNKNOWN",
     "but see below: it is a residual we can read off the market"),
], columns=["quantity", "status", "evidence"])
display(known)

**The one unknown is observable, and that is the step that ends the argument.**

Market inventory obeys exactly one identity per good and per turn:

$$
v_{g,t+1} - v_{g,t} \;=\; \underbrace{q_{g,t}}_{\text{ours, known}} \;+\; \underbrace{s_{g,t}}_{\text{theirs}} \;-\; \underbrace{d_{g,t}}_{\text{town}}
$$

We issue $q_{g,t}$ ourselves, so rearranging isolates the whole of what we would otherwise have to forecast in two separate pieces:

$$
\pi_{g,t} \;\equiv\; s_{g,t} - d_{g,t} \;=\; \big(v_{g,t+1} - v_{g,t}\big) - q_{g,t}
$$

Call $\pi_g$ the **external pressure**. It is not a model of anything: it is a subtraction of two numbers the observation already contains. Positive means the market is filling faster than the town drains it, so a unit is worth more now than later; negative means the reverse and holding pays. We smooth it with an exponential moving average, and that is the entire opponent model.

That turns the per-turn decision into a marginal comparison with no free threshold in it. Sell the $k$-th unit of good $g$ exactly while it is worth at least what the *first* unit would fetch $H$ turns from now:

$$
\text{sell unit } k
\quad\Longleftrightarrow\quad
p_g\big(v_{g,t} + k\big) \;\ge\; p_g\big(v_{g,t} + \hat{\pi}_{g,t}\, H\big)
$$

**No price gate and no rate cap survive that**: both are replaced by a quantity the observation already contains. It is the classical marginal-revenue stopping rule. Section 1's concavity is what makes the stopping point well defined at all, with the same caveat as there: optimal for the single-good relaxation, not for the constrained problem we actually run.

So we never have to model the opponent, or the shop draw, or the town clock separately. **One subtraction replaces all three**, and it needs an estimator, not a policy.

**What that leaves for RL to do: estimate by sampling what can be computed.** The field's own evidence points the same way, when a perfect simulator exists, search with a learned evaluation dominates model-free learning by orders of magnitude in samples (Silver et al., 2017; and Sutton, 1990, for the model-based argument in its original form).

**And the budget arithmetic is not close.** One episode costs about a second in our harness. A model-free run of 10⁶ episodes is roughly 28 hours of pure rollout on ten cores *before* any tuning, and the competition closes on 2026-09-30. The deployed agent is also stdlib-only with a one-second turn budget, so whatever is learned has to compile down to a table or a handful of coefficients, which is exactly the object the search in section 5 produces directly.

**[C] Where learning would earn its keep is the one place with no model: the opponent's reaction function.** And there the right family is *imitation*, not RL, because a replay stores the full observation of both seats at every turn. Their input and their output are both recorded (Pomerleau, 1991; Ross, Gordon & Bagnell, 2011, for the distribution shift that behavioural cloning walks into).

## 3. What are we maximising? The obvious answer is measurably wrong

Our bank. Except that both farms draw from the same purse in the figure above, so a unit we decline to sell is a unit sold at a higher price **by the opponent**.

We built the naive version first and measured it. Its own bank went **up** and it lost catastrophically.

In [ ]:
duel = pd.DataFrame([
    ("first game measured, parent", 68_232,  60_268),
    ("first game measured, restrained seller", 73_586, 176_163),
], columns=["configuration", "our bank", "their bank"])
duel["margin"] = duel["our bank"] - duel["their bank"]
display(duel)

rate = pd.DataFrame([(3, 79_862, 88_844), (40, 75_811, 77_078)],
                    columns=["units per item per turn", "our bank", "their bank"])
rate["margin"] = rate["our bank"] - rate["their bank"]
display(rate)
print("Selling harder costs us $4,051 and costs the opponent $11,766.")

The restrained seller **banked $5,354 more than the route it replaces** and turned a $7,964 win into a $102,577 loss. Pooled over twenty matchups it lifted the *opponent's* median bank from $80,421 to $145,902.

**A seller optimised on its own bank optimises the other farm's too.** This is the practical content of *Dueling Algorithms* (Immorlica, Kalai, Lucier, Moitra, Postlewaite & Tennenholtz, STOC 2011) at the level of a single order: in a duel what is maximised is P(beat the opponent), not E[value], and here the two point in opposite directions.

It also means the classical result from section 1 has to be read carefully. Selling **last** maximises absolute revenue, because inventory is lowest after the town has eaten and before anyone has sold. That is correct, and it is correct about the wrong quantity.

## 4. So the objective is the win rate, and here the derivation overreached

Section 3 is solidly supported: the score is the *margin*, not the bank, and the effect is enormous.

The next step looked like it followed. A duel maximises P(win), the win rate over twenty games is a step function that Nelder-Mead cannot walk on, so replace it with `mean(tanh(margin / S))`, smooth, saturating, and equal to the win rate in the limit. We made that change during the search and justified it with an observation made **in passing**: on one evaluation, a configuration with a mean margin of −1,495 took 8 wins of 20, while another at −990 took only 4.

**One evaluation is an anecdote.** Before writing it down as a reason, we measured it: 80 configurations from a Latin hypercube over the same box the search uses, so the population is the one the search actually sees.

In [ ]:
OBJ = pd.DataFrame([{"mean": -4075.9, "wins": 3, "tanh": -0.5392740557869817}, {"mean": -6906.85, "wins": 2, "tanh": -0.6432665052125875}, {"mean": -7390.35, "wins": 2, "tanh": -0.6772425796233669}, {"mean": -13200.35, "wins": 1, "tanh": -0.8946506471351393}, {"mean": -13343.5, "wins": 1, "tanh": -0.897597841295033}, {"mean": -8845.15, "wins": 1, "tanh": -0.8302435253947179}, {"mean": -40574.05, "wins": 0, "tanh": -0.9999611832310247}, {"mean": -10230.45, "wins": 2, "tanh": -0.8155852456880346}, {"mean": -36197.5, "wins": 0, "tanh": -0.9999939161436711}, {"mean": -597.7, "wins": 9, "tanh": -0.11903997368828956}, {"mean": -4943.5, "wins": 2, "tanh": -0.5907181549802462}, {"mean": -18449.85, "wins": 0, "tanh": -0.9673898133324619}, {"mean": -1355.9, "wins": 8, "tanh": -0.22294735011614933}, {"mean": -8256.6, "wins": 1, "tanh": -0.8563702797329981}, {"mean": -28071.2, "wins": 0, "tanh": -0.9988248570932896}, {"mean": -2944.15, "wins": 4, "tanh": -0.42312083324289995}, {"mean": -1259.6, "wins": 4, "tanh": -0.2829884182228754}, {"mean": -27679.3, "wins": 0, "tanh": -0.9995003169068444}, {"mean": -1962.4, "wins": 4, "tanh": -0.38765268375253936}, {"mean": -3527.4, "wins": 4, "tanh": -0.48587379932920516}, {"mean": -4760.1, "wins": 3, "tanh": -0.5909031877149981}, {"mean": -23052.2, "wins": 0, "tanh": -0.9917945068309176}, {"mean": -409.05, "wins": 9, "tanh": -0.11101672737786183}, {"mean": -12882.4, "wins": 0, "tanh": -0.9482381626922576}, {"mean": -17111.65, "wins": 0, "tanh": -0.9694602738739686}, {"mean": -3920.7, "wins": 4, "tanh": -0.4936473193639254}, {"mean": -25997.5, "wins": 0, "tanh": -0.9983283432885589}, {"mean": -1709.15, "wins": 5, "tanh": -0.2985202191703994}, {"mean": -27587.75, "wins": 0, "tanh": -0.998063799184284}, {"mean": -2343.0, "wins": 3, "tanh": -0.42345032921799186}, {"mean": -7676.65, "wins": 2, "tanh": -0.7004562956294178}, {"mean": -3522.25, "wins": 4, "tanh": -0.48577496084318766}, {"mean": -9153.05, "wins": 1, "tanh": -0.8465678616434168}, {"mean": -9474.85, "wins": 1, "tanh": -0.8496683095507775}, {"mean": -13912.65, "wins": 1, "tanh": -0.8993191989795124}, {"mean": -30346.75, "wins": 0, "tanh": -0.9998064794264749}, {"mean": -5140.95, "wins": 2, "tanh": -0.6115730396928061}, {"mean": -7822.6, "wins": 2, "tanh": -0.6638833769919779}, {"mean": -24920.75, "wins": 0, "tanh": -0.99853099281872}, {"mean": -11688.65, "wins": 1, "tanh": -0.8846306158308239}, {"mean": -78749.95, "wins": 0, "tanh": -0.9999999957732989}, {"mean": -13926.8, "wins": 1, "tanh": -0.9074578244627409}, {"mean": -77702.1, "wins": 0, "tanh": -0.9999999999146999}, {"mean": -17437.4, "wins": 1, "tanh": -0.9405878729080202}, {"mean": -9380.35, "wins": 1, "tanh": -0.8814431835261136}, {"mean": -3611.0, "wins": 4, "tanh": -0.469621748189416}, {"mean": -1382.35, "wins": 8, "tanh": -0.23054810397633885}, {"mean": -5044.55, "wins": 2, "tanh": -0.6014933965430938}, {"mean": -4889.25, "wins": 2, "tanh": -0.6264356743571782}, {"mean": -8327.6, "wins": 1, "tanh": -0.8535303934982542}, {"mean": -3697.45, "wins": 4, "tanh": -0.47849240184060315}, {"mean": -5364.2, "wins": 2, "tanh": -0.676222882958947}, {"mean": -9388.45, "wins": 1, "tanh": -0.8348178770580004}, {"mean": -7820.25, "wins": 2, "tanh": -0.7251463507046024}, {"mean": -10538.65, "wins": 1, "tanh": -0.857710063113853}, {"mean": -4391.0, "wins": 2, "tanh": -0.5552462588995428}, {"mean": -16447.95, "wins": 0, "tanh": -0.9734476806504062}, {"mean": -21604.6, "wins": 0, "tanh": -0.9836424702944113}, {"mean": -20041.35, "wins": 0, "tanh": -0.9708281087609445}, {"mean": -1355.05, "wins": 8, "tanh": -0.21965509988862494}, {"mean": -27981.1, "wins": 0, "tanh": -0.9988264100538131}, {"mean": -26618.9, "wins": 0, "tanh": -0.9979148887723877}, {"mean": -26466.6, "wins": 0, "tanh": -0.9993754032538895}, {"mean": -306.55, "wins": 9, "tanh": -0.08974057961681874}, {"mean": -22053.55, "wins": 0, "tanh": -0.9914687072770432}, {"mean": -4540.55, "wins": 2, "tanh": -0.575419720373231}, {"mean": -930.3, "wins": 8, "tanh": -0.20596912561935107}, {"mean": -864.75, "wins": 6, "tanh": -0.21371128376026358}, {"mean": -26879.95, "wins": 0, "tanh": -0.994213232582767}, {"mean": -495.35, "wins": 8, "tanh": -0.1336900668336121}, {"mean": -23862.75, "wins": 0, "tanh": -0.9905994186088739}, {"mean": -22480.3, "wins": 0, "tanh": -0.9886702535938247}, {"mean": -10900.45, "wins": 1, "tanh": -0.8624228434313336}, {"mean": -7642.15, "wins": 2, "tanh": -0.6982351752124989}, {"mean": -6819.1, "wins": 2, "tanh": -0.6869407328214823}, {"mean": -13483.75, "wins": 1, "tanh": -0.9026269790743466}, {"mean": -11684.05, "wins": 1, "tanh": -0.8847816164688673}, {"mean": -13387.15, "wins": 1, "tanh": -0.8980139154803023}, {"mean": -6667.45, "wins": 2, "tanh": -0.7182777987133069}, {"mean": -37150.3, "wins": 0, "tanh": -0.9999445041313757}])
SCALE = 4000.0
N_MATCH = 20
print(f"{len(OBJ)} configurations, {N_MATCH} matchups each "
      f"(research/objective_disagreement.py)")

def spearman(a, b):
    ra, rb = pd.Series(a).rank(), pd.Series(b).rank()
    return float(ra.corr(rb))

print(f"\nSpearman rank correlation with the win rate")
print(f"  mean margin     {spearman(OBJ['mean'], OBJ['wins']):+.3f}")
print(f"  tanh surrogate  {spearman(OBJ['tanh'], OBJ['wins']):+.3f}")

pick_mean = OBJ['mean'].idxmax(); pick_tanh = OBJ['tanh'].idxmax()
pick_wins = OBJ['wins'].idxmax()
print(f"\nThe configuration each objective would choose")
for name, i in [("mean margin", pick_mean), ("tanh surrogate", pick_tanh),
                ("win rate", pick_wins)]:
    print(f"  by {name:<15} {OBJ.wins[i]:>2}/{N_MATCH} wins, "
          f"mean margin {OBJ['mean'][i]:>+9,.0f}")
print(f"\n  optimising the mean instead of the win rate costs "
      f"{OBJ.wins[pick_wins] - OBJ.wins[pick_mean]} wins")

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 4.6))
sc = ax.scatter(OBJ["mean"], OBJ["wins"], s=44, c=OBJ["tanh"], cmap="viridis",
                edgecolor="white", linewidth=0.5, zorder=3)
for i, col, name in [(pick_mean, RED, "picked by mean margin"),
                     (pick_wins, TEAL, "picked by win rate")]:
    ax.scatter([OBJ["mean"][i]], [OBJ["wins"][i]], s=220, facecolor="none",
               edgecolor=col, linewidth=2.4, zorder=4)
    ax.annotate(name, (OBJ["mean"][i], OBJ["wins"][i]),
                textcoords="offset points", xytext=(8, 10), fontsize=8.5,
                color=col, fontweight="bold")
ax.set_xlabel("mean paired margin over the matchups ($)")
ax.set_ylabel(f"games won of {N_MATCH}")
ax.set_title("The two objectives order the same configurations after all", loc="left")
cb = fig.colorbar(sc, ax=ax); cb.set_label("tanh surrogate", fontsize=9)
plt.tight_layout(); plt.show()

**They agree, and the refinement buys nothing.** The mean margin ranks configurations at ρ = +0.96 against the win rate, the surrogate at ρ = +0.97, and all three objectives pick a configuration with the *same* number of wins. The cost of having optimised the mean instead is **zero games**.

So the step is retracted as a *finding*, and kept as a *choice*. The objective in the search is still

$$\frac{1}{n}\sum_{g} \tanh\!\left(\frac{\text{margin}_g}{S}\right)$$

with $S = \$4{,}000$, because it is smooth, saturating, and tends to the win rate as $S \to 0$, and because the duel argument for optimising P(win) is sound independently of whether it changed anything here. What is **not** true is that it mattered on this surface, and the first draft of this notebook said it did.

**The lesson is one this project has now paid for four times in different clothes.** Two episodes is not evidence about an agent. One evaluation is not evidence about an objective. A pair of points that invert is exactly what a rank correlation of 0.96 produces, inversions are not counter-examples, they are the residual.

**What survives from section 3 is much larger and is not in doubt**: optimising the bank instead of the margin cost the entire win rate, 0 of 20 against the parent's 20 of 20, and handed the opponent $115,895 in a single game. The objective question that mattered was *relative versus absolute*, and it was settled before this one was asked.

## 5. What does the surface look like? Because that picks the optimiser

The reflex is to sweep one parameter at a time. We did, and the sweep measured its own inadequacy.

In [ ]:
SWEEP = pd.DataFrame([{"label": "MODE=greedy", "wins": 8, "n": 20, "median_margin": -1761.0, "median_bank": 75098.0, "median_theirs": 77603.5}, {"label": "MODE=mpc", "wins": 3, "n": 20, "median_margin": -1699.0, "median_bank": 75348.5, "median_theirs": 78389.0}, {"label": "PROJECT=0", "wins": 0, "n": 20, "median_margin": -80435.5, "median_bank": 53726.0, "median_theirs": 134610.5}, {"label": "PROJECT=1", "wins": 3, "n": 20, "median_margin": -1699.0, "median_bank": 75348.5, "median_theirs": 78389.0}, {"label": "FUND=0", "wins": 1, "n": 20, "median_margin": -4786.5, "median_bank": 78071.5, "median_theirs": 84949.5}, {"label": "FUND=1", "wins": 3, "n": 20, "median_margin": -1699.0, "median_bank": 75348.5, "median_theirs": 78389.0}, {"label": "FUND_H=1", "wins": 1, "n": 20, "median_margin": -4791.5, "median_bank": 78111.5, "median_theirs": 84947.5}, {"label": "FUND_H=4", "wins": 2, "n": 20, "median_margin": -3571.5, "median_bank": 78345.0, "median_theirs": 84572.5}, {"label": "FUND_H=24", "wins": 3, "n": 20, "median_margin": -1699.0, "median_bank": 75348.5, "median_theirs": 78389.0}, {"label": "FUND_H=72", "wins": 3, "n": 20, "median_margin": -1699.0, "median_bank": 75348.5, "median_theirs": 78389.0}, {"label": "FUND_H=240", "wins": 3, "n": 20, "median_margin": -1699.0, "median_bank": 75349.5, "median_theirs": 78389.0}, {"label": "HORIZON=12", "wins": 3, "n": 20, "median_margin": -2089.5, "median_bank": 75322.5, "median_theirs": 78602.5}, {"label": "HORIZON=24", "wins": 3, "n": 20, "median_margin": -1944.0, "median_bank": 75346.0, "median_theirs": 78494.5}, {"label": "HORIZON=48", "wins": 3, "n": 20, "median_margin": -1699.0, "median_bank": 75348.5, "median_theirs": 78389.0}, {"label": "HORIZON=96", "wins": 3, "n": 20, "median_margin": -1575.0, "median_bank": 75339.5, "median_theirs": 78316.5}, {"label": "HORIZON=240", "wins": 3, "n": 20, "median_margin": -1575.0, "median_bank": 75323.0, "median_theirs": 78316.0}, {"label": "PRESSURE_W=0.03", "wins": 4, "n": 20, "median_margin": -1613.0, "median_bank": 75317.5, "median_theirs": 78390.0}, {"label": "PRESSURE_W=0.1", "wins": 3, "n": 20, "median_margin": -1699.0, "median_bank": 75348.5, "median_theirs": 78389.0}, {"label": "PRESSURE_W=0.3", "wins": 3, "n": 20, "median_margin": -1419.5, "median_bank": 75374.5, "median_theirs": 78170.5}, {"label": "RESERVE_H=12", "wins": 0, "n": 20, "median_margin": -35448.0, "median_bank": 61594.5, "median_theirs": 98385.0}, {"label": "RESERVE_H=24", "wins": 6, "n": 20, "median_margin": -2383.0, "median_bank": 76378.0, "median_theirs": 79667.5}, {"label": "RESERVE_H=48", "wins": 3, "n": 20, "median_margin": -1699.0, "median_bank": 75348.5, "median_theirs": 78389.0}, {"label": "RESERVE_H=96", "wins": 1, "n": 20, "median_margin": -14233.5, "median_bank": 79576.5, "median_theirs": 94646.0}, {"label": "SHED_MARGIN=4", "wins": 3, "n": 20, "median_margin": -1879.5, "median_bank": 75348.5, "median_theirs": 78398.0}, {"label": "SHED_MARGIN=12", "wins": 3, "n": 20, "median_margin": -1699.0, "median_bank": 75348.5, "median_theirs": 78389.0}, {"label": "SHED_MARGIN=30", "wins": 8, "n": 20, "median_margin": -986.0, "median_bank": 75437.5, "median_theirs": 78093.5}, {"label": "ROUTE_TURNS_ONLY=0", "wins": 3, "n": 20, "median_margin": -1699.0, "median_bank": 75348.5, "median_theirs": 78389.0}, {"label": "ROUTE_TURNS_ONLY=1", "wins": 5, "n": 20, "median_margin": -1709.0, "median_bank": 75610.0, "median_theirs": 78120.0}, {"label": "SELLS_FIRST=0", "wins": 0, "n": 20, "median_margin": -95852.0, "median_bank": 40150.5, "median_theirs": 138161.5}, {"label": "SELLS_FIRST=1", "wins": 3, "n": 20, "median_margin": -1699.0, "median_bank": 75348.5, "median_theirs": 78389.0}, {"label": "FLUSH_AT=648", "wins": 3, "n": 20, "median_margin": -2490.5, "median_bank": 74287.5, "median_theirs": 78500.0}, {"label": "FLUSH_AT=696", "wins": 3, "n": 20, "median_margin": -1699.0, "median_bank": 75348.5, "median_theirs": 78389.0}, {"label": "FLUSH_AT=999", "wins": 3, "n": 20, "median_margin": -1699.0, "median_bank": 75305.0, "median_theirs": 78389.0}, {"label": "CASH_MIN=0", "wins": 3, "n": 20, "median_margin": -1699.0, "median_bank": 75348.5, "median_theirs": 78389.0}, {"label": "CASH_MIN=400", "wins": 3, "n": 20, "median_margin": -1699.0, "median_bank": 75348.5, "median_theirs": 78389.0}, {"label": "CASH_MIN=2000", "wins": 3, "n": 20, "median_margin": -1699.0, "median_bank": 75348.5, "median_theirs": 78389.0}, {"label": "FUND_MARGIN=1.0", "wins": 3, "n": 20, "median_margin": -1699.0, "median_bank": 75348.5, "median_theirs": 78389.0}, {"label": "FUND_MARGIN=1.3", "wins": 3, "n": 20, "median_margin": -1699.0, "median_bank": 75348.5, "median_theirs": 78389.0}, {"label": "FUND_MARGIN=2.0", "wins": 3, "n": 20, "median_margin": -1699.0, "median_bank": 75348.5, "median_theirs": 78389.0}])
res = SWEEP[SWEEP.label.str.startswith("RESERVE_H")].copy()
res["turns"] = res.label.str.split("=").str[1].astype(int)
display(res[["label", "wins", "n", "median_margin"]])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.0),
                         gridspec_kw={"width_ratios": [1.15, 1]})

ax = axes[0]
d = res.sort_values("turns")
ax.plot(d.turns, d.wins, "-o", color=TEAL, lw=2.2, ms=8, zorder=3)
for _, r in d.iterrows():
    ax.annotate(f"{int(r.wins)}/{int(r.n)}", (r.turns, r.wins),
                textcoords="offset points", xytext=(0, 10), ha="center",
                fontsize=8.5, color=INK)
ax.axhspan(-0.4, 1.5, color=RED, alpha=0.07)
ax.text(60, 0.6, "the plan starves / nothing is ever free to sell",
        fontsize=8.5, color=RED, ha="center")
ax.set_xlabel("reserve horizon (turns of the plan's own demand held back)")
ax.set_ylabel("games won of 20")
ax.set_ylim(-0.5, 15)
ax.set_title("Two cliffs, not a slope", loc="left")

ax = axes[1]
# The parent's own row is in this table at 20/20 and is not a configuration of
# the seller. Excluding it is the difference between "the best a single axis
# reaches" and "the number we are trying to beat".
best_axis = SWEEP[~SWEEP.label.str.startswith("parent")].wins.max()
bars = [("best any single\naxis reaches", int(best_axis), GREY),
        ("one hypercube point\nmoving three at once", 12, TEAL)]
ax.bar([0, 1], [b[1] for b in bars], color=[b[2] for b in bars], width=0.55)
ax.set_xticks([0, 1]); ax.set_xticklabels([b[0] for b in bars], fontsize=9)
for i, b in enumerate(bars):
    ax.text(i, b[1] + 0.25, f"{b[1]}/20", ha="center", fontweight="bold")
ax.set_ylim(0, 15); ax.set_ylabel("games won of 20")
ax.set_title("A ridge a coordinate sweep cannot see", loc="left")
plt.tight_layout(); plt.show()

**The reserve horizon has two cliffs and a plateau between them.** Too little and the plan starves. It sells the wheat its own animals eat. Too much and nothing is ever free to sell.

And the axes are not separable: the best value **any** single axis reaches on its own is worse than a hypercube point that moves three of them together. A coordinate sweep cannot see a ridge running diagonally between two cliffs, and reading its best row as "the setting" is reading a projection as the surface.

**That is the standard case for derivative-free optimisation**, and the standard advice for it, about a dozen parameters, an evaluation costing seconds, a non-differentiable objective with cliffs, is a global stage followed by a local one (Rios & Sahinidis, 2013). Ours is a Latin hypercube, which stratifies every one-dimensional projection instead of letting uniform sampling clump (McKay, Beckman & Conover, 1979), then Nelder-Mead restarts from the best points (Nelder & Mead, 1965). CMA-ES (Hansen & Ostermeier, 2001) would be the other reasonable choice; at this budget the optimiser is not what decides the answer.

## 6. And the gate that matters more than the optimiser

Search a dozen parameters against twenty games, then quote the winner's score, and you are quoting a number that was selected for being high.

So the matchups split into two disjoint halves stratified by seat. **The entire search sees only one half.** The result is reported on the other, which it has never been evaluated against once.

In [ ]:
curse = pd.DataFrame([
    ("parent, screen",  20, 5991),
    ("best, screen",    14, 786),
    ("parent, CONFIRM", 19, 3840),
    ("best, CONFIRM",   10, -308),
], columns=["", "wins of 20", "mean margin"])
display(curse)
drop = 100 * (14 - 10) / 20
print(f"screen-to-confirm drop: {drop:.0f} points, over 259 evaluations")

**Twenty points.** That is the winner's curse, measured rather than assumed, and it is the largest this project has recorded. A search that reported only its screen number would have claimed 14 of 20; the honest number is 10.

Which is also the result: **the searched seller does not beat the recorded schedule it replaces.** In dollars it is a coin flip, half the games land within $117, while the route wins nineteen of twenty by about $4,400 each. Winning small and reliably is exactly what a duel pays for, and the recording does it.

## 6b. What the ecosystem does to the measurement, not just to the agent

Everything above optimises against a fixed set of recorded matchups. That is the right instrument for *did this break the route, and does it move the margin against the field we actually met*. It is the wrong one for *will this rate higher*, and the reason is section 0's third result: the environment turns over with a half life of about a week.

Three consequences, and each one is a rule we now measure by rather than a caveat.

**A pool is a sample of a population that is dying.** So an evaluation pool is admitted by two measured properties, what share of current field seats it covers, and how many distinct construction plans it contains, and never by its age. A pool built this morning from a dying clade is exactly as unrepresentative as one built last week.

**A rating is a measurement of an agent against the field of that moment, never a property of the agent.** We have this from the inside, twice, on our own submissions while byte-identical: one fell from a peak of 2,913.7 to 2,731.9 over 135 episodes, about **−2.1 points per episode**; another fell 2,849.4 → 2,693.1 over two days. Both falls are larger than any gap we had been reading as a ranking.

**And a retired submission's rating freezes** where it stopped receiving episodes, so drift only acts on what is still playing. A table of past agents therefore flatters whoever stopped earliest, and its order is roughly the inverse of exposure.

$$
\text{rating}(A, t) \;=\; \text{quality}(A) \;\otimes\; \text{field}(t)
\qquad\Longrightarrow\qquad
\text{rating}(A, t_1) \;\not\gtrless\; \text{rating}(B, t_2)
$$

**The only comparison that survives is the concurrent one**: two submissions in the play window, facing the same field, read in the same pass. That is not a refinement of the measurement. It is the design constraint that decides what may be fielded and when.

*This is the seam where this notebook ends and the instrument notebook begins. Everything here is about choosing the search; that one is about whether the number the search returns can be believed.*

## 7. What we are testing right now, and how it may be read

The searched agent is on the ladder. The offline harness makes a sharp prediction: clearly behind its parent.

**So the two submissions in the play window are read together, or not at all**, §6b is why. Neither is comparable to any historical number, including our own.

If the ladder agrees with the offline instrument, the instrument is validated on a case it called *against itself*. If it disagrees, the instrument is broken, and we need to know that more than we need the rating.

## 8. What would change the method

**If the plan were not frozen.** Generating construction rather than replaying it is combinatorial over 720 turns and 100 tiles, and heavy search earns its keep there.

**If the opponent adapted inside our horizon.** They are invisible until turn 136 in this game, which is a fifth of the season, and that bounds how much an opponent model can be worth. Change that and the calculus changes.

**If the price curve moved.** Everything in section 1 is engine constants. A rebalance retunes every threshold silently, which is why the agent adopts base prices from the observation on turn 0, where inventory is exactly `I0` and the quoted price *is* the base, and checks its own model against the quote every turn, falling back when they disagree.

---

## References

**The shape of the problem**
- Ibaraki, T. & Katoh, N. (1988). *Resource Allocation Problems: Algorithmic Approaches.* MIT Press. Greedy marginal allocation is optimal for separable concave objectives under a single resource constraint.
- Bertsimas, D. & Lo, A. W. (1998). Optimal control of execution costs. *Journal of Financial Markets* 1(1), 1-50.
- Almgren, R. & Chriss, N. (2000). Optimal execution of portfolio transactions. *Journal of Risk* 3(2), 5-39. Closed form for *linear* impact; ours is not linear, so the structure transfers and the formula does not.
- Mayne, D. Q., Rawlings, J. B., Rao, C. V. & Scokaert, P. O. M. (2000). Constrained model predictive control: stability and optimality. *Automatica* 36(6), 789-814.

**The objective**
- Immorlica, N., Kalai, A. T., Lucier, B., Moitra, A., Postlewaite, A. & Tennenholtz, M. (2011). Dueling Algorithms. *STOC 2011*, 215-224.
- Cournot, A. (1838). *Recherches sur les principes mathématiques de la théorie des richesses.* The oversupply reading of a market both players flood.

**Planning against learning**
- Sutton, R. S. (1990). Integrated architectures for learning, planning and reacting based on approximating dynamic programming. *ICML*.
- Silver, D. et al. (2017). Mastering the game of Go without human knowledge. *Nature* 550, 354-359.
- Pomerleau, D. A. (1991). Efficient training of artificial neural networks for autonomous navigation. *Neural Computation* 3(1), 88-97.
- Ross, S., Gordon, G. & Bagnell, J. A. (2011). A reduction of imitation learning and structured prediction to no-regret online learning. *AISTATS*.

**The search**
- Rios, L. M. & Sahinidis, N. V. (2013). Derivative-free optimization: a review of algorithms and comparison of software implementations. *Journal of Global Optimization* 56(3), 1247-1293.
- McKay, M. D., Beckman, R. J. & Conover, W. J. (1979). A comparison of three methods for selecting values of input variables in the analysis of output from a computer code. *Technometrics* 21(2), 239-245.
- Nelder, J. A. & Mead, R. (1965). A simplex method for function minimization. *Computer Journal* 7(4), 308-313.
- Hansen, N. & Ostermeier, A. (2001). Completely derandomized self-adaptation in evolution strategies. *Evolutionary Computation* 9(2), 159-195.

*Volumes and page numbers are written from memory and worth checking before anyone quotes them. The titles and years are the load-bearing part.*